# Module 3b: Production Deployment with Failure Cases

## Overview

This notebook deploys an **intentionally broken version** of the Product Catalog Agent from Module 03. It contains 4 failure cases that represent real-world production issues:

1. **Poor Tool Metadata + Routing Mismatch** - Swapped descriptions AND swapped Lambda routing cause wrong data to be returned (e.g., asking for product details returns only stock info)
2. **Tool Parameter Schema Mismatch** - Type validation errors from unclear parameter descriptions
3. **Wrong Model ID** - Deprecated model causing invocation failures (latent — masked by env var override)
4. **Missing Conversation History** - Stateless agent with no memory across turns

Your mission: deploy this broken agent, observe the failures using CloudWatch + X-Ray, diagnose root causes, and apply fixes.

### Important: Independent Prefix

This deployment uses a **different prefix** (`ecommerce-workshop-broken`) to avoid overwriting the working agent from Module 03. Both agents coexist side by side.

### Prerequisites

- Module 03 completed (Cognito User Pool, DynamoDB tables already exist)
- Docker installed and running

### Time: ~15 minutes (deployment only)

## Step 1: Environment Setup

Load the working deployment config from Module 03 and set up the broken variant prefix.

In [1]:
import boto3
import json
import os
import time
import uuid
import subprocess

# Load region and deployment info from Module 03
try:
    %store -r REGION
    %store -r deployment_info
    print(f"Loaded REGION from Module 03: {REGION}")
    print(f"Working agent runtime: {deployment_info.get('runtime_id', 'N/A')}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    deployment_info = {}
    print(f"Could not load Module 03 state, using default region: {REGION}")

os.environ['AWS_REGION'] = REGION
os.environ['AWS_DEFAULT_REGION'] = REGION

# Broken variant uses a different prefix to coexist with the working agent
from deployment_config import WORKSHOP_PREFIX, RUNTIME_NAME

# Module directory — this notebook lives alongside the broken agent code
MODULE_DIR = os.path.dirname(os.path.abspath('__file__')) if os.path.exists('agents') else '.'
PRODUCTS_TABLE = 'ecommerce-workshop-products'  # Reuses existing DynamoDB table

# Naming conventions (all use the broken prefix)
LAMBDA_ROLE_NAME = f'{WORKSHOP_PREFIX}-lambda-role'
GATEWAY_ROLE_NAME = f'{WORKSHOP_PREFIX}-gateway-role'
GATEWAY_NAME = f'{WORKSHOP_PREFIX}-product-gateway'
RUNTIME_ROLE_NAME = f'{WORKSHOP_PREFIX}-runtime-role'
ECR_REPO_NAME = f'{WORKSHOP_PREFIX}-product-catalog-agent'
TEST_PASSWORD = 'Workshop1234!'

# AWS clients
sts = boto3.client('sts', region_name=REGION)
identity = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']

print(f"\nConfiguration:")
print(f"  Region: {REGION}")
print(f"  AWS Account: {ACCOUNT_ID}")
print(f"  Broken Prefix: {WORKSHOP_PREFIX}")
print(f"  Runtime Name: {RUNTIME_NAME}")
print(f"  Products Table: {PRODUCTS_TABLE} (shared with working agent)")
print(f"  Module Dir: {MODULE_DIR}")

Loaded REGION from Module 03: us-west-2
Working agent runtime: ecommerce_workshop_product_catalog_agent-Nda58K37yj

Configuration:
  Region: us-west-2
  AWS Account: 534409838809
  Broken Prefix: ecommerce-workshop-broken
  Runtime Name: ecommerce_workshop_broken_product_catalog_agent
  Products Table: ecommerce-workshop-products (shared with working agent)
  Module Dir: /Users/mmelli/Documents/projects/claude-code-agentcore/ecommerce-agent-workshop/03-production-deployment-with-failure-cases


## Step 2: Reuse Cognito from Module 03

The broken agent reuses the same Cognito User Pool, groups, and test users from Module 03. No need to recreate them.

In [2]:
from utils import (
    get_or_create_cognito_user_pool,
    get_or_create_cognito_resource_server,
    get_or_create_cognito_app_client,
    get_or_create_user_app_client,
    get_or_create_cognito_domain,
    get_user_token,
    get_oauth_token
)

cognito_client = boto3.client('cognito-idp', region_name=REGION)

# Reuse the existing Cognito User Pool from Module 03
COGNITO_POOL_NAME = 'ecommerce-workshop-user-pool'  # Original prefix
USER_POOL_ID = get_or_create_cognito_user_pool(cognito_client, COGNITO_POOL_NAME)

# Create a separate M2M client for this broken gateway
COGNITO_RESOURCE_SERVER_ID = f'{WORKSHOP_PREFIX}-gateway-api'
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access to Gateway tools"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access to Gateway tools"}
]
get_or_create_cognito_resource_server(
    cognito_client, USER_POOL_ID, COGNITO_RESOURCE_SERVER_ID,
    f"{WORKSHOP_PREFIX} Gateway API", SCOPES
)

M2M_CLIENT_NAME = f'{WORKSHOP_PREFIX}-gateway-client'
M2M_CLIENT_ID, M2M_CLIENT_SECRET = get_or_create_cognito_app_client(
    cognito_client, USER_POOL_ID, M2M_CLIENT_NAME, COGNITO_RESOURCE_SERVER_ID, SCOPES
)

# Reuse the existing user app client from Module 03
USER_CLIENT_NAME = 'ecommerce-workshop-user-client'  # Original prefix
USER_CLIENT_ID = get_or_create_user_app_client(
    cognito_client, USER_POOL_ID, USER_CLIENT_NAME
)

# Cognito discovery URL
COGNITO_DISCOVERY_URL = f'https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration'

# Test user credentials (same as Module 03)
CUSTOMER_EMAIL = 'john.customer@example.com'
ADMIN_EMAIL = 'alice.admin@example.com'

print(f"Cognito Configuration:")
print(f"  User Pool ID: {USER_POOL_ID} (reused from Module 03)")
print(f"  M2M Client ID: {M2M_CLIENT_ID} (new for broken gateway)")
print(f"  User Client ID: {USER_CLIENT_ID} (reused from Module 03)")
print(f"  Test Customer: {CUSTOMER_EMAIL}")
print(f"  Test Admin: {ADMIN_EMAIL}")

Found existing user pool: us-west-2_zZFgcKaqx
Resource server ecommerce-workshop-broken-gateway-api already exists
Found existing app client: 69jus17omaavptitssm702i13p
Found existing user app client: 3nvfo8a3bfvipd9k07pfi37mnf
Updated auth flows on user app client
Cognito Configuration:
  User Pool ID: us-west-2_zZFgcKaqx (reused from Module 03)
  M2M Client ID: 69jus17omaavptitssm702i13p (new for broken gateway)
  User Client ID: 3nvfo8a3bfvipd9k07pfi37mnf (reused from Module 03)
  Test Customer: john.customer@example.com
  Test Admin: alice.admin@example.com


## Step 3: Create IAM Roles and Deploy Lambda Functions

Create separate IAM roles and Lambda functions for the broken agent. These use the broken prefix to avoid conflicts.

In [28]:
from utils import create_lambda_execution_role, create_lambda_function

iam_client = boto3.client('iam', region_name=REGION)
lambda_client = boto3.client('lambda', region_name=REGION)

# DynamoDB table ARN (shared table)
products_table_arn = f'arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{PRODUCTS_TABLE}'

# Create Lambda execution role
print("Creating Lambda execution role...")
lambda_role_resp = create_lambda_execution_role(
    iam_client, LAMBDA_ROLE_NAME, [products_table_arn]
)
LAMBDA_ROLE_ARN = lambda_role_resp['Role']['Arn']
print(f"Lambda Role ARN: {LAMBDA_ROLE_ARN}")

# Environment variables
lambda_env_vars = {'PRODUCTS_TABLE_NAME': PRODUCTS_TABLE}

# Deploy Product Tools Lambda (with intentional failures)
print("\nDeploying Product Tools Lambda (with failures)...")
product_tools_result = create_lambda_function(
    lambda_client,
    f'{WORKSHOP_PREFIX}-product-tools',
    LAMBDA_ROLE_ARN,
    f'{MODULE_DIR}/lambda_tools/product_tools_lambda.py',
    'product_tools_lambda.lambda_handler',
    lambda_env_vars,
    REGION
)
PRODUCT_TOOLS_ARN = product_tools_result['function_arn']
print(f"Product Tools ARN: {PRODUCT_TOOLS_ARN}")

# Deploy RBAC Interceptor Lambda (unchanged from working version)
print("\nDeploying RBAC Interceptor Lambda...")
interceptor_result = create_lambda_function(
    lambda_client,
    f'{WORKSHOP_PREFIX}-rbac-interceptor',
    LAMBDA_ROLE_ARN,
    f'{MODULE_DIR}/lambda_tools/rbac_interceptor_lambda.py',
    'rbac_interceptor_lambda.lambda_handler',
    {},
    REGION
)
INTERCEPTOR_ARN = interceptor_result['function_arn']
print(f"RBAC Interceptor ARN: {INTERCEPTOR_ARN}")

Creating Lambda execution role...
Role ecommerce-workshop-broken-lambda-role already exists, retrieving ARN...
Lambda Role ARN: arn:aws:iam::534409838809:role/ecommerce-workshop-broken-lambda-role

Deploying Product Tools Lambda (with failures)...
Function ecommerce-workshop-broken-product-tools exists, updating...
Product Tools ARN: arn:aws:lambda:us-west-2:534409838809:function:ecommerce-workshop-broken-product-tools

Deploying RBAC Interceptor Lambda...
Function ecommerce-workshop-broken-rbac-interceptor exists, updating...
RBAC Interceptor ARN: arn:aws:lambda:us-west-2:534409838809:function:ecommerce-workshop-broken-rbac-interceptor


## Step 4: Create AgentCore Gateway

Create a separate Gateway for the broken agent with JWT auth and RBAC interceptor.

In [36]:
from utils import create_agentcore_gateway_role, create_gateway, create_lambda_gateway_target, get_product_tool_schemas

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Create Gateway IAM role
print("Creating Gateway IAM role...")
gateway_role_resp = create_agentcore_gateway_role(
    iam_client, GATEWAY_ROLE_NAME, [PRODUCT_TOOLS_ARN, INTERCEPTOR_ARN]
)
GATEWAY_ROLE_ARN = gateway_role_resp['Role']['Arn']
print(f"Gateway Role ARN: {GATEWAY_ROLE_ARN}")

# Create Gateway with JWT auth
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [M2M_CLIENT_ID, USER_CLIENT_ID],
        "discoveryUrl": COGNITO_DISCOVERY_URL
    }
}

print("\nCreating AgentCore Gateway with RBAC interceptor...")
gateway_response = create_gateway(
    gateway_client, GATEWAY_NAME, GATEWAY_ROLE_ARN, auth_config,
    'Broken Product Catalog Gateway for failure case workshop',
    interceptor_lambda_arn=INTERCEPTOR_ARN
)

if gateway_response:
    GATEWAY_ID = gateway_response['gatewayId']
    GATEWAY_URL = gateway_response['gatewayUrl']
    GATEWAY_ARN = f'arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}'
    print(f"\nGateway Created:")
    print(f"  Gateway ID: {GATEWAY_ID}")
    print(f"  Gateway URL: {GATEWAY_URL}")
else:
    print("ERROR: Gateway creation failed")

# Reload utils to pick up updated tool schemas
import importlib
import utils
importlib.reload(utils)
from utils import get_product_tool_schemas

Creating Gateway IAM role...
Role ecommerce-workshop-broken-gateway-role already exists, updating policy...
Updated Lambda invoke policy on Gateway role
Gateway Role ARN: arn:aws:iam::534409838809:role/ecommerce-workshop-broken-gateway-role

Creating AgentCore Gateway with RBAC interceptor...
Gateway 'ecommerce-workshop-broken-product-gateway' already exists, retrieving...
Found existing Gateway: ecommerce-workshop-broken-product-gateway-8cayfahgf8

Gateway Created:
  Gateway ID: ecommerce-workshop-broken-product-gateway-8cayfahgf8
  Gateway URL: https://ecommerce-workshop-broken-product-gateway-8cayfahgf8.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp


In [37]:
# Wait for Gateway to reach ACTIVE status before registering targets
print("Waiting for Gateway to be ready...")
max_wait = 120
poll_interval = 10
waited = 0
gateway_ready = False

while waited < max_wait:
    try:
        gw_info = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
        gw_status = gw_info.get('status', 'UNKNOWN')
        print(f"  Gateway status: {gw_status} (waited {waited}s)")
        if gw_status in ('ACTIVE', 'READY'):
            gateway_ready = True
            break
    except Exception as e:
        print(f"  Error checking gateway: {e}")
    time.sleep(poll_interval)
    waited += poll_interval

if not gateway_ready:
    print(f"WARNING: Gateway not ACTIVE after {max_wait}s. Target creation may fail.")

# Delete existing target first so schema changes take effect
print("\nRemoving existing ProductTools target (if any) to refresh schemas...")
try:
    existing_targets = gateway_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
    for t in existing_targets.get('items', existing_targets.get('targets', [])):
        tid = t.get('targetId', t.get('name', ''))
        tname = t.get('name', tid)
        if tname == 'ProductTools' or 'ProductTools' in tid:
            gateway_client.delete_gateway_target(
                gatewayIdentifier=GATEWAY_ID, targetId=tid
            )
            print(f"  Deleted target: {tname} ({tid})")
            time.sleep(5)  # Brief wait for deletion to propagate
except Exception as e:
    print(f"  (No existing target to delete or error: {e})")

# Register product tools as Gateway target
# NOTE: get_product_tool_schemas() returns the BROKEN schemas (from local utils.py)
# - "search" has vague description "Find items"
# - get_product_details and check_inventory have SWAPPED descriptions
# - update_inventory has new_quantity typed as STRING (should be integer)
# Additionally, the Lambda TOOLS dict has SWAPPED routing:
# - "get_product_details" → check_inventory() (returns stock only)
# - "check_inventory" → get_product_details() (returns full details)
TOOL_SCHEMAS = get_product_tool_schemas()
print(f"\nRegistering {len(TOOL_SCHEMAS)} product tools as Gateway target:")
print(f"(These schemas contain intentional failures - Failure Cases 1 & 2)\n")
for schema in TOOL_SCHEMAS:
    print(f"  - {schema['name']}: {schema['description'][:80]}...")

print(f"\nCreating Gateway target...")
product_target = create_lambda_gateway_target(
    gateway_client, GATEWAY_ID, 'ProductTools',
    PRODUCT_TOOLS_ARN, TOOL_SCHEMAS,
    'Product catalog tools (with intentional failures)'
)

if product_target:
    print(f"Target created: ProductTools")
    print(f"Target ID: {product_target.get('targetId')}")
else:
    print("ERROR: Failed to create Gateway target")

Waiting for Gateway to be ready...
  Gateway status: READY (waited 0s)

Removing existing ProductTools target (if any) to refresh schemas...
  Deleted target: ProductTools (XHGIXAFQPR)

Registering 11 product tools as Gateway target:
(These schemas contain intentional failures - Failure Cases 1 & 2)

  - search: Find items...
  - get_product_details: Check current stock levels and availability for a product...
  - check_inventory: Get detailed product information including name, price, description, and specifi...
  - get_product_recommendations: Get product recommendations based on category and price criteria...
  - compare_products: Get detailed product information including name, price, description, and specifi...
  - get_return_policy: Get return policy information, optionally for a specific product...
  - create_product: Create a new product in the catalog (admin only)...
  - update_product: Update an existing product's information (admin only)...
  - delete_product: Soft-delete a 

## Step 5: Build and Push Agent Container

Build the broken agent container image and push to ECR. The container uses the broken `product_catalog_agent.py` which contains Failure Cases 3 (wrong model ID) and 4 (stateless agent).

In [6]:
# Create ECR repository
ecr_client = boto3.client('ecr', region_name=REGION)

print("Creating ECR repository...")
try:
    ecr_client.create_repository(
        repositoryName=ECR_REPO_NAME,
        imageScanningConfiguration={'scanOnPush': True},
        imageTagMutability='MUTABLE'
    )
    print(f"Created: {ECR_REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"Exists: {ECR_REPO_NAME}")

ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
CONTAINER_URI = f"{ECR_REGISTRY}/{ECR_REPO_NAME}:latest"
print(f"Container URI: {CONTAINER_URI}")

# Login to ECR
print("\nLogging into ECR...")
login_cmd = f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ECR_REGISTRY}"
result = subprocess.run(login_cmd, shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("ECR login successful")
else:
    print(f"ECR login failed: {result.stderr}")

Creating ECR repository...
Created: ecommerce-workshop-broken-product-catalog-agent
Container URI: 534409838809.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-broken-product-catalog-agent:latest

Logging into ECR...
ECR login successful


In [7]:
# Build Docker image (ARM64 required by AgentCore Runtime)
print("Building Docker image (ARM64) with broken agent code...")
print("This may take a few minutes on first build.\n")

build_cmd = f"docker build --platform linux/arm64 -t {ECR_REPO_NAME}:latest {MODULE_DIR}/agents/"
result = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print("Build successful!")
else:
    print(f"Build failed:\n{result.stderr}")

Building Docker image (ARM64) with broken agent code...
This may take a few minutes on first build.

Build successful!


In [8]:
# Tag and push to ECR
print("Pushing to ECR...")

tag_cmd = f"docker tag {ECR_REPO_NAME}:latest {CONTAINER_URI}"
result = subprocess.run(tag_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print(f"Tag failed: {result.stderr}")
else:
    push_cmd = f"docker push {CONTAINER_URI}"
    result = subprocess.run(push_cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Pushed successfully: {CONTAINER_URI}")
    else:
        print(f"Push failed:\n{result.stderr}")

Pushing to ECR...
Pushed successfully: 534409838809.dkr.ecr.us-west-2.amazonaws.com/ecommerce-workshop-broken-product-catalog-agent:latest


## Step 6: Create AgentCore Runtime

Deploy the broken agent container to AgentCore Runtime with OpenTelemetry observability enabled.

In [18]:
from utils import create_agent_runtime_role, create_agent_runtime

agentcore_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Create Runtime IAM role
print("Creating AgentCore Runtime role...")
runtime_role_resp = create_agent_runtime_role(
    iam_client, RUNTIME_ROLE_NAME, gateway_arn=GATEWAY_ARN
)
RUNTIME_ROLE_ARN = runtime_role_resp['Role']['Arn']
print(f"Runtime Role ARN: {RUNTIME_ROLE_ARN}")

# OTEL service name
OTEL_SERVICE_NAME = RUNTIME_NAME.replace('_', '-')

# Agent environment variables (includes OTEL config)
# NOTE: MODEL_ID is passed here so the agent functions. However, the agent code
# has a broken default ("anthropic.claude-3-5-haiku-20241022-v1:0") that would
# cause failures if this env var were ever removed — that is Failure Case 3.
agent_env_vars = {
    'AGENT_REGION': REGION,
    'GATEWAY_URL': GATEWAY_URL,
    'MODEL_ID': 'global.anthropic.claude-sonnet-4-6',
    'AGENT_OBSERVABILITY_ENABLED': 'true',
    'OTEL_PYTHON_DISTRO': 'aws_distro',
    'OTEL_PYTHON_CONFIGURATOR': 'aws_configurator',
    'OTEL_TRACES_EXPORTER': 'otlp',
    'OTEL_LOGS_EXPORTER': 'otlp',
    'OTEL_METRICS_EXPORTER': 'none',
    'OTEL_EXPORTER_OTLP_PROTOCOL': 'http/protobuf',
    'OTEL_SERVICE_NAME': OTEL_SERVICE_NAME,
    'OTEL_EXPORTER_OTLP_TRACES_ENDPOINT': f'https://xray.{REGION}.amazonaws.com/v1/traces',
    'OTEL_EXPORTER_OTLP_LOGS_ENDPOINT': f'https://logs.{REGION}.amazonaws.com/v1/logs',
    'OTEL_SEMCONV_STABILITY_OPT_IN': 'gen_ai_tool_definitions',
}

print(f"\nDeploying broken agent to AgentCore Runtime...")
print(f"This may take several minutes while the container starts up.\n")

runtime_response = create_agent_runtime(
    agentcore_client,
    runtime_name=RUNTIME_NAME,
    role_arn=RUNTIME_ROLE_ARN,
    container_uri=CONTAINER_URI,
    environment_vars=agent_env_vars,
    description='Broken Product Catalog Agent for failure case workshop'
)

if runtime_response:
    RUNTIME_ID = runtime_response['agentRuntimeId']
    RUNTIME_ARN = runtime_response['agentRuntimeArn']
    print(f"\nRuntime deployed:")
    print(f"  Runtime ID: {RUNTIME_ID}")
    print(f"  Runtime ARN: {RUNTIME_ARN}")
    print(f"  Status: {runtime_response.get('status')}")
else:
    # Try to recover from saved config
    print("\nRuntime creation returned None. Attempting to recover from saved config...")
    try:
        with open('deployment_config.json') as _f:
            _saved = json.load(_f)
        RUNTIME_ID = _saved['runtime_id']
        RUNTIME_ARN = _saved['runtime_arn']
        print(f"  Recovered RUNTIME_ID: {RUNTIME_ID}")
        print(f"  Recovered RUNTIME_ARN: {RUNTIME_ARN}")
    except (FileNotFoundError, KeyError):
        raise RuntimeError(
            "Runtime deployment failed and no saved config found. "
            "Check AWS credentials (mwinit) and clean up any stale resources before retrying."
        )

Creating AgentCore Runtime role...
Role ecommerce-workshop-broken-runtime-role already exists, updating policies...
Updated gateway policy on runtime role
Ensured X-Ray tracing policy on runtime role
Runtime Role ARN: arn:aws:iam::534409838809:role/ecommerce-workshop-broken-runtime-role

Deploying broken agent to AgentCore Runtime...
This may take several minutes while the container starts up.

Created AgentCore Runtime: ecommerce_workshop_broken_product_catalog_agent
Runtime ARN: arn:aws:bedrock-agentcore:us-west-2:534409838809:runtime/ecommerce_workshop_broken_product_catalog_agent-Z5sd5uBTBd
Waiting for runtime to be ready...
  Status: CREATING...
Runtime is ready!

Runtime deployed:
  Runtime ID: ecommerce_workshop_broken_product_catalog_agent-Z5sd5uBTBd
  Runtime ARN: arn:aws:bedrock-agentcore:us-west-2:534409838809:runtime/ecommerce_workshop_broken_product_catalog_agent-Z5sd5uBTBd
  Status: READY


## Step 7: Get Test User Tokens

Obtain JWT tokens for both customer and admin test users.

In [19]:
import base64

# Get tokens for both test users
print("Getting JWT tokens for test users...\n")

customer_tokens = get_user_token(
    cognito_client, USER_POOL_ID, USER_CLIENT_ID,
    CUSTOMER_EMAIL, TEST_PASSWORD
)

admin_tokens = get_user_token(
    cognito_client, USER_POOL_ID, USER_CLIENT_ID,
    ADMIN_EMAIL, TEST_PASSWORD
)

def decode_jwt_claims(token):
    parts = token.split('.')
    payload_b64 = parts[1]
    padding = 4 - len(payload_b64) % 4
    if padding != 4:
        payload_b64 += '=' * padding
    return json.loads(base64.urlsafe_b64decode(payload_b64))

if customer_tokens.get('id_token'):
    claims = decode_jwt_claims(customer_tokens['id_token'])
    print(f"Customer: {claims.get('email')} (groups: {claims.get('cognito:groups', [])})")

if admin_tokens.get('id_token'):
    claims = decode_jwt_claims(admin_tokens['id_token'])
    print(f"Admin: {claims.get('email')} (groups: {claims.get('cognito:groups', [])})")

Getting JWT tokens for test users...

Got tokens for user: john.customer@example.com
Got tokens for user: alice.admin@example.com
Customer: john.customer@example.com (groups: ['customer'])
Admin: alice.admin@example.com (groups: ['admin'])


## Step 8: Test the Broken Agent

Now test the broken agent with scenarios designed to trigger each failure case. Use the observability tools (CloudWatch Logs, X-Ray traces) to diagnose what went wrong.

See `FAILURECASES.md` for the full list of test scenarios and diagnosis steps.

In [38]:
import importlib
import utils
importlib.reload(utils)
from utils import invoke_agent_runtime

agentcore_runtime_client = boto3.client('bedrock-agentcore', region_name=REGION)

# --- Failure Case 1: Poor Tool Metadata ---
# The search tool has a vague description ("Find items") and the Lambda routing
# is SWAPPED: get_product_details calls check_inventory() and vice versa.
# Combined with misleading descriptions, the agent returns wrong/incomplete data.

customer_session_id = f"broken-customer-{str(uuid.uuid4())}"

print("=" * 60)
print("FAILURE CASE 1: Poor Tool Metadata")
print("Test: Product Search (vague tool description)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client, RUNTIME_ARN, customer_session_id,
    {
        'prompt': 'Search for wireless headphones under $100',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
tools_used = result.get('metadata', {}).get('tools_used', [])
if tools_used:
    print(f"Tools used: {tools_used}")
print(f"Response:\n{result.get('response', result.get('error', 'No response'))}")

FAILURE CASE 1: Poor Tool Metadata
Test: Product Search (vague tool description)

Status: success
Tools used: ['search', 'get_product_recommendations']
Response:
Unfortunately, it looks like our search and recommendations came up empty for **wireless headphones under $100** at this time. Here's what might help:

- 🔍 **Try broader terms** — I can search for just "headphones" or "audio" if you'd like to see what's available in that space.
- 💡 **Adjust your budget** — There may be wireless headphones available at a slightly higher price point.
- 📂 **Browse other categories** — If you're open to related products like earbuds or speakers, I can search those too.

Would you like me to try any of these alternatives? Just let me know how you'd like to proceed!


In [39]:
# --- Failure Case 1 (continued): Swapped Lambda routing ---
# get_product_details routes to check_inventory() which returns ONLY stock info.
# The agent will return incomplete data (missing price, specs, features, etc.)

print("=" * 60)
print("FAILURE CASE 1: Swapped Tool Routing + Misleading Descriptions")
print("Test: Product Details (get_product_details → check_inventory)")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client, RUNTIME_ARN, customer_session_id,
    {
        'prompt': 'Show me details about product PROD-001',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': customer_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
tools_used = result.get('metadata', {}).get('tools_used', [])
if tools_used:
    print(f"Tools used: {tools_used}")
print(f"Response:\n{result.get('response', result.get('error', 'No response'))}")
print(f"\nExpected failure: Response should be MISSING price, description,")
print(f"features, specifications, and warranty because get_product_details")
print(f"is routed to check_inventory() which only returns stock info.")

FAILURE CASE 1: Swapped Tool Routing + Misleading Descriptions
Test: Product Details (get_product_details → check_inventory)

Status: success
Tools used: ['get_product_details']
Response:
Here are the details for **PROD-001**:

- 🎧 **Product Name:** Wireless Bluetooth Headphones
- ✅ **Availability:** In Stock and Ready to Ship
- 📦 **Quantity Available:** 100 units

Would you like me to:
- Get **more detailed information** (price, description, specifications)?
- Check the **return policy** for this product?
- **Compare** it with other products?

Just let me know how I can help! 😊

Expected failure: Response should be MISSING price, description,
features, specifications, and warranty because get_product_details
is routed to check_inventory() which only returns stock info.


In [40]:
# --- Failure Case 2: Tool Parameter Schema Mismatch ---
# The update_inventory tool has a vague parameter description for new_quantity.
# LLM may pass strings like "one hundred units" instead of an integer.

admin_session_id = f"broken-admin-{str(uuid.uuid4())}"

print("=" * 60)
print("FAILURE CASE 2: Tool Parameter Schema Mismatch")
print("Test: Update inventory with ambiguous quantity")
print("=" * 60)

result = invoke_agent_runtime(
    agentcore_runtime_client, RUNTIME_ARN, admin_session_id,
    {
        'prompt': 'Set inventory for PROD-001 to one hundred units',
        'bearer_token': admin_tokens.get('id_token', ''),
        'access_token': admin_tokens.get('access_token', ''),
        'session_id': admin_session_id
    }
)

print(f"\nStatus: {result.get('status')}")
tools_used = result.get('metadata', {}).get('tools_used', [])
if tools_used:
    print(f"Tools used: {tools_used}")
print(f"Response:\n{result.get('response', result.get('error', 'No response'))}")
print(f"\n(Check CloudWatch Lambda logs for TypeError on new_quantity)")

FAILURE CASE 2: Tool Parameter Schema Mismatch
Test: Update inventory with ambiguous quantity

Status: success
Tools used: ['update_inventory', 'update_inventory']
Response:
The update is unfortunately failing with a backend error — specifically, it appears there's a type mismatch issue on the server side (`'<' not supported between instances of 'str' and 'int'`), which is preventing the inventory from being updated.

Here's a summary of what happened:
- **Product:** PROD-001
- **Requested Quantity:** 100 units
- **Status:** ❌ Failed — Backend error

This appears to be a **system-level bug** rather than an issue with the request itself. I'd recommend:
1. **Reporting this to your development/engineering team** to investigate the type comparison error in the inventory update logic.
2. **Trying again later** once the issue has been resolved.

Would you like help with anything else in the meantime?

(Check CloudWatch Lambda logs for TypeError on new_quantity)


In [42]:
# --- Failure Case 3: Missing Conversation History ---
# The agent is stateless — no memory across turns.

multiturn_session_id = f"broken-multiturn-{str(uuid.uuid4())}"

print("=" * 60)
print("FAILURE CASE 3: Missing Conversation History")
print("Test: Multi-turn conversation (agent should lose context)")
print("=" * 60)

# Turn 1
print("\n--- Turn 1: Ask about a specific product ---")
result1 = invoke_agent_runtime(
    agentcore_runtime_client, RUNTIME_ARN, multiturn_session_id,
    {
        'prompt': 'Show me details about product PROD-001',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': multiturn_session_id
    }
)
tools_used_1 = result1.get('metadata', {}).get('tools_used', [])
print(f"Tools used: {tools_used_1}")
print(f"Response: {str(result1.get('response', ''))[:200]}...")

time.sleep(2)

# Turn 2: Follow-up that requires context from Turn 1
print("\n--- Turn 2: Follow-up (requires context from Turn 1) ---")
result2 = invoke_agent_runtime(
    agentcore_runtime_client, RUNTIME_ARN, multiturn_session_id,
    {
        'prompt': 'Is that product in stock?',
        'bearer_token': customer_tokens.get('id_token', ''),
        'access_token': customer_tokens.get('access_token', ''),
        'session_id': multiturn_session_id
    }
)
tools_used_2 = result2.get('metadata', {}).get('tools_used', [])
print(f"Tools used: {tools_used_2}")
print(f"Response: {str(result2.get('response', ''))[:200]}...")

print(f"\n(If the agent asks 'Which product?' or doesn't know about PROD-001,")
print(f" that confirms Failure Case 4 — stateless agent with no conversation history)")

FAILURE CASE 3: Missing Conversation History
Test: Multi-turn conversation (agent should lose context)

--- Turn 1: Ask about a specific product ---
Tools used: ['get_product_details']
Response: Here are the details for **PROD-001**:

| Field | Details |
|---|---|
| 🎧 **Product Name** | Wireless Bluetooth Headphones |
| 🆔 **Product ID** | PROD-001 |
| ✅ **Availability** | In Stock & Ready to ...

--- Turn 2: Follow-up (requires context from Turn 1) ---
Tools used: []
Response: It looks like you haven't mentioned a specific product yet! Could you please provide me with the product name or product ID you'd like me to check stock availability for? I'll be happy to look that up...

(If the agent asks 'Which product?' or doesn't know about PROD-001,
 that confirms Failure Case 4 — stateless agent with no conversation history)


## Step 9: Save Configuration

Save the broken agent configuration for the Streamlit app and downstream use.

In [ ]:
from utils import save_config

# Save config for Streamlit app
agent_config = {
    'runtime_arn': RUNTIME_ARN,
    'runtime_id': RUNTIME_ID,
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'region': REGION,
    'user_pool_id': USER_POOL_ID,
    'user_client_id': USER_CLIENT_ID,
    'customer_email': CUSTOMER_EMAIL,
    'admin_email': ADMIN_EMAIL,
    'test_password': TEST_PASSWORD,
    'otel_service_name': OTEL_SERVICE_NAME,
    'variant': 'broken'  # Marks this as the failure-case variant
}

save_config(agent_config, 'streamlit_app/agent_config.json')

# Also save deployment info for recovery
broken_deployment_info = {
    'runtime_arn': RUNTIME_ARN,
    'runtime_id': RUNTIME_ID,
    'runtime_name': RUNTIME_NAME,
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'user_pool_id': USER_POOL_ID,
    'user_client_id': USER_CLIENT_ID,
    'otel_service_name': OTEL_SERVICE_NAME,
    'region': REGION,
    'prefix': WORKSHOP_PREFIX
}

with open('deployment_config.json', 'w') as f:
    json.dump(broken_deployment_info, f, indent=2)

%store broken_deployment_info

print(f"Configuration saved.")
print(f"\nTo launch the Streamlit app:")
print(f"  cd streamlit_app")
print(f"  streamlit run app.py")

In [ ]:
print("=" * 60)
print("BROKEN AGENT DEPLOYMENT COMPLETE")
print("=" * 60)

print(f"\nInfrastructure (prefix: {WORKSHOP_PREFIX}):")
print(f"  Runtime: {RUNTIME_ARN}")
print(f"  Gateway: {GATEWAY_URL}")
print(f"  Cognito: {USER_POOL_ID} (shared with working agent)")

print(f"\nFailure Cases Deployed:")
print(f"  1. Poor Tool Metadata - vague/identical tool descriptions")
print(f"  2. Parameter Schema Mismatch - type validation errors")
print(f"  3. Wrong Model ID - deprecated model")
print(f"  4. Missing Conversation History - stateless agent")

print(f"\nObservability:")
print(f"  OTEL Service: {OTEL_SERVICE_NAME}")
print(f"  CloudWatch: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#genai-observability:bedrockAgentCore")

print(f"\nNext Steps:")
print(f"  1. Test each failure case via Streamlit or the cells above")
print(f"  2. Use CloudWatch Logs + X-Ray to diagnose root causes")
print(f"  3. Read FAILURECASES.md for detailed diagnosis guides")
print(f"  4. Apply fixes, rebuild, and redeploy to verify")
print("=" * 60)

---

## Cleanup (Optional)

Run this cell to delete all broken-variant resources. The working agent from Module 03 is not affected.

In [ ]:
# UNCOMMENT AND RUN TO CLEAN UP BROKEN AGENT RESOURCES
#
# # --- Recover variables if kernel was restarted ---
# try:
#     %store -r broken_deployment_info
#     RUNTIME_ID = broken_deployment_info['runtime_id']
#     GATEWAY_ID = broken_deployment_info['gateway_id']
#     REGION = broken_deployment_info['region']
#     print("Recovered variables from %store")
# except:
#     try:
#         with open('deployment_config.json') as f:
#             _info = json.load(f)
#         RUNTIME_ID = _info['runtime_id']
#         GATEWAY_ID = _info['gateway_id']
#         REGION = _info.get('region', 'us-west-2')
#         print("Recovered variables from deployment_config.json")
#     except FileNotFoundError:
#         print("ERROR: No saved state found. Please set RUNTIME_ID and GATEWAY_ID manually.")

# from deployment_config import WORKSHOP_PREFIX, RUNTIME_NAME

# os.environ['AWS_REGION'] = REGION
# os.environ['AWS_DEFAULT_REGION'] = REGION

# from utils import delete_agent_runtime, delete_gateway

# iam_client = boto3.client('iam', region_name=REGION)
# lambda_client = boto3.client('lambda', region_name=REGION)
# ecr_client = boto3.client('ecr', region_name=REGION)
# gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
# agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)

# print(f"Cleaning up broken agent resources (prefix: {WORKSHOP_PREFIX})...\n")

# # 1. Delete Runtime
# print("Deleting AgentCore Runtime...")
# delete_agent_runtime(agentcore_client, RUNTIME_ID)

# # 2. Delete Gateway
# print("\nDeleting AgentCore Gateway...")
# delete_gateway(gateway_client, GATEWAY_ID)

# # 3. Delete Lambda functions
# print("\nDeleting Lambda functions...")
# for func_name in [f'{WORKSHOP_PREFIX}-product-tools', f'{WORKSHOP_PREFIX}-rbac-interceptor']:
#     try:
#         lambda_client.delete_function(FunctionName=func_name)
#         print(f"  Deleted: {func_name}")
#     except Exception as e:
#         print(f"  Error deleting {func_name}: {e}")

# # 4. Delete ECR repository
# ECR_REPO_NAME = f'{WORKSHOP_PREFIX}-product-catalog-agent'
# print("\nDeleting ECR repository...")
# try:
#     ecr_client.delete_repository(repositoryName=ECR_REPO_NAME, force=True)
#     print(f"  Deleted: {ECR_REPO_NAME}")
# except Exception as e:
#     print(f"  Error: {e}")

# # 5. Delete IAM roles
# print("\nDeleting IAM roles...")
# for role_name in [f'{WORKSHOP_PREFIX}-lambda-role', f'{WORKSHOP_PREFIX}-gateway-role', f'{WORKSHOP_PREFIX}-runtime-role']:
#     try:
#         policies = iam_client.list_attached_role_policies(RoleName=role_name)
#         for policy in policies['AttachedPolicies']:
#             iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy['PolicyArn'])
#         inline = iam_client.list_role_policies(RoleName=role_name)
#         for policy_name in inline['PolicyNames']:
#             iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
#         iam_client.delete_role(RoleName=role_name)
#         print(f"  Deleted: {role_name}")
#     except Exception as e:
#         print(f"  Error deleting {role_name}: {e}")

# print("\nCleanup complete! (Working agent from Module 03 is unaffected)")